To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

Read our **[Qwen3 Guide](https://docs.unsloth.ai/basics/qwen3-how-to-run-and-fine-tune)** and check out our new **[Dynamic 2.0](https://docs.unsloth.ai/basics/unsloth-dynamic-2.0-ggufs)** quants which outperforms other quantization methods!

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

### Unsloth

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.5.6: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.5.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


<a name="Data"></a>
### Data Prep
We now use the Alpaca dataset from [yahma](https://huggingface.co/datasets/yahma/alpaca-cleaned), which is a filtered version of 52K of the original [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html). You can replace this code section with your own data prep.

**[NOTE]** To train only on completions (ignoring the user's input) read TRL's docs [here](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only).

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output!! Otherwise you'll get infinite generations!

If you want to use the `llama-3` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Alpaca.ipynb)

For text completions like novel writing, try this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_(7B)-Text_Completion.ipynb).

In [37]:
from google.colab import files
uploaded = files.upload()


Saving train_generated_samples.json to train_generated_samples.json


In [38]:
import json
from datasets import Dataset

with open("train_generated_samples.json", "r", encoding="utf-8") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

In [39]:
alpaca_prompt = """### Instruction:
بر اساس علائم و شکایات زیر، تشخیص بیماری را بده.

### Input:
{}

### Response:
{}"""

def format_dataset(example):
    instruction = "بر اساس علائم و شکایات زیر، تشخیص بیماری را بده."
    input_text = example["symptom"]
    output_text = f"{example['diagnosis'][0]} ({example['diagnosis'][1]})"  #fas+eng
    text = alpaca_prompt.format(input_text, output_text)
    return {"text": text}

# converting the dataset to Alpaca format
dataset = dataset.map(format_dataset)


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [40]:
print(dataset[0])


{'symptom': 'مردی ۵۰ ساله با کاهش ناگهانی بینایی در چشم راست و سردرد مراجعه کرده است.', 'diagnosis': ['انسداد جریان خون شبکیه', 'Retinal Artery Occlusion'], 'text': '### Instruction:\nبر اساس علائم و شکایات زیر، تشخیص بیماری را بده.\n\n### Input:\nمردی ۵۰ ساله با کاهش ناگهانی بینایی در چشم راست و سردرد مراجعه کرده است.\n\n### Response:\nانسداد جریان خون شبکیه (Retinal Artery Occlusion)'}


<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [41]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 5, # Set this for 1 full training run.
        # max_steps = 60,
        learning_rate = 2e-4,
        # learning_rate = 5e-5,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

In [25]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
7.135 GB of memory reserved.


In [42]:
trainer_stats = trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 5 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040/8,000,000,000 (0.52% trained)


Step,Training Loss
1,0.270600
2,0.245100
3,0.286900
4,0.324100
5,0.347800
6,0.370200
7,0.324300
8,0.395500
9,0.356100
10,0.351500


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!



In [43]:
# Define the instruction-based prompt
alpaca_prompt = """### Instruction:
بر اساس علائم و شکایات زیر، تشخیص بیماری را بده.

### Input:
{}

### Response:
"""

# Enable inference mode (optimized for unsloth models)
FastLanguageModel.for_inference(model)

# Example symptom input
symptom_input = "مردی ۴۰ ساله با سرفه خشک و کاهش وزن مراجعه کرده است."

# Tokenize the prompt with your symptom
inputs = tokenizer(
    [alpaca_prompt.format(symptom_input)],
    return_tensors="pt"
).to("cuda")

# Generate prediction
outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)

# Decode and print result
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])


### Instruction:
بر اساس علائم و شکایات زیر، تشخیص بیماری را بده.

### Input:
مردی ۴۰ ساله با سرفه خشک و کاهش وزن مراجعه کرده است.

### Response:
سرطان ریه (Lung Cancer)


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [44]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

###**Test dataset and evaluation**

In [45]:
from google.colab import files
uploaded = files.upload()

Saving test_generated_samples.json to test_generated_samples.json


In [46]:
import json

with open("test_generated_samples.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)


In [47]:
alpaca_test_prompt = """### Instruction:
بر اساس سوال زیر، گزینه صحیح را از میان گزینه‌ها انتخاب کن.

### Input:
سوال: {question}
گزینه‌ها: {options}

### Response:
"""


In [32]:
# import difflib
# import csv

# # قالب پرامپت مشابه زمان آموزش
# alpaca_test_prompt = """### Instruction:
# بر اساس علائم و شکایات زیر، تشخیص بیماری را بده.

# ### Input:
# {}

# ### Response:
# """

# correct = 0
# total = len(test_data)
# results = []

# for sample in test_data:
#     question = sample["question"]
#     options = sample["options"]
#     answer = sample["answer"]

#     # تبدیل صورت سؤال به prompt مدل
#     prompt = alpaca_test_prompt.format(question)

#     # اجرای مدل روی prompt
#     inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
#     outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
#     raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

#     # پاک‌سازی خروجی خام برای استخراج بیماری
#     if "تشخیص داده شده است" in raw_output:
#         raw_output = raw_output.split("تشخیص داده شده است")[0]
#     if "(" in raw_output:
#         raw_output = raw_output.split("(")[0]
#     if ":" in raw_output:
#         raw_output = raw_output.split(":")[-1]
#     cleaned_output = raw_output.strip()

#     # پیدا کردن نزدیک‌ترین گزینه
#     closest_matches = difflib.get_close_matches(cleaned_output, options, n=1, cutoff=0.3)
#     predicted_answer = closest_matches[0] if closest_matches else "هیچ‌کدام"

#     if predicted_answer == answer:
#         correct += 1

#     # ذخیره در لیست خروجی‌ها
#     results.append({
#         "سوال": question,
#         "گزینه‌ها": " | ".join(options),
#         "پاسخ صحیح": answer,
#         "پیش‌بینی مدل": predicted_answer,
#         "خروجی خام مدل": raw_output
#     })

# # محاسبه دقت نهایی
# accuracy = correct / total
# print(f" دقت نهایی مدل: {accuracy:.2%}")

# # ذخیره نتایج در فایل CSV
# with open("model_predictions.csv", "w", newline='', encoding="utf-8-sig") as f:
#     writer = csv.DictWriter(f, fieldnames=["سوال", "گزینه‌ها", "پاسخ صحیح", "پیش‌بینی مدل", "خروجی خام مدل"])
#     writer.writeheader()
#     writer.writerows(results)

# print("فایل CSV با تمام نتایج در 'model_predictions.csv' ذخیره شد.")


 دقت نهایی مدل: 66.00%
فایل CSV با تمام نتایج در 'model_predictions.csv' ذخیره شد.


In [50]:
import difflib
import csv

alpaca_test_prompt = """### Instruction:
بر اساس علائم و شکایات زیر، تشخیص بیماری را بده.

### Input:
{}

### Response:
"""

correct = 0
total = len(test_data)
results = []

for sample in test_data:
    question = sample["question"]
    options = sample["options"]
    answer = sample["answer"]

    prompt = alpaca_test_prompt.format(question)

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
    raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    if "تشخیص داده شده است" in raw_output:
        raw_output = raw_output.split("تشخیص داده شده است")[0]
    if "(" in raw_output:
        raw_output = raw_output.split("(")[0]
    if ":" in raw_output:
        raw_output = raw_output.split(":")[-1]
    cleaned_output = raw_output.strip()

    closest_matches = difflib.get_close_matches(cleaned_output, options, n=1, cutoff=0.3)
    predicted_answer = closest_matches[0] if closest_matches else "هیچ‌کدام"

    if predicted_answer == answer:
        correct += 1

    results.append({
        "سوال": question,
        "گزینه‌ها": " | ".join(options),
        "پاسخ صحیح": answer,
        "پیش‌بینی مدل": predicted_answer,
        "خروجی خام مدل": cleaned_output
    })

accuracy = correct / total
print(f" دقت نهایی مدل: {accuracy:.2%}")

with open("model_predictions.csv", "w", newline='', encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["سوال", "گزینه‌ها", "پاسخ صحیح", "پیش‌بینی مدل", "خروجی خام مدل"])
    writer.writeheader()
    writer.writerows(results)

print("فایل CSV با تمام نتایج در 'model_predictions.csv' ذخیره شد.")


 دقت نهایی مدل: 74.00%
فایل CSV با تمام نتایج در 'model_predictions.csv' ذخیره شد.
